# Session 12: Putting It All Together - Mini Project

Capstone project combining all skills: data loading, cleaning, analysis, and visualization

**Duration:** 30 minutes

## Project: Personal Fitness Tracker Analysis

We'll analyze fitness data using everything we've learned.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timedelta

%matplotlib inline

## Step 1: Create Sample Data

In [ ]:
# Generate 30 days of fitness data
np.random.seed(42)
days = 30
start_date = datetime(2024, 1, 1)

fitness_data = {
    'Date': [start_date + timedelta(days=i) for i in range(days)],
    'Steps': np.random.randint(5000, 15000, days),
    'Calories': np.random.randint(1800, 2800, days),
    'Exercise_Minutes': np.random.randint(0, 90, days),
    'Sleep_Hours': np.random.uniform(5, 9, days).round(1),
    'Weight_lbs': 180 - np.cumsum(np.random.uniform(-0.3, 0.3, days)).round(1),
    'Water_Glasses': np.random.randint(4, 12, days)
}

df = pd.DataFrame(fitness_data)
df['DayOfWeek'] = df['Date'].dt.day_name()

print('Sample Data:')
print(df.head())
print(f'\nDataset shape: {df.shape}')

## Step 2: Data Cleaning

In [ ]:
# Check for issues
print('Data Info:')
print(df.info())
print('\nMissing Values:')
print(df.isnull().sum())
print('\nBasic Statistics:')
print(df.describe())

## Step 3: Add Calculated Columns

In [ ]:
# Create categories and metrics
df['Steps_Category'] = pd.cut(df['Steps'], 
                              bins=[0, 7500, 10000, 20000],
                              labels=['Low', 'Good', 'Excellent'])

df['Active_Day'] = df['Exercise_Minutes'] >= 30
df['Well_Hydrated'] = df['Water_Glasses'] >= 8
df['Good_Sleep'] = df['Sleep_Hours'] >= 7

print('Enhanced Data:')
print(df[['Date', 'Steps_Category', 'Active_Day', 'Well_Hydrated', 'Good_Sleep']].head())

## Step 4: Analysis

In [ ]:
def generate_fitness_report(df):
    """Generate comprehensive fitness report."""
    print('='*60)
    print('PERSONAL FITNESS REPORT')
    print('='*60)
    
    # Overall stats
    print(f'\n📊 Period: {df["Date"].min().date()} to {df["Date"].max().date()}')
    print(f'Total Days Tracked: {len(df)}')
    
    # Activity metrics
    print(f'\n🚶 Activity Metrics:')
    print(f'   Average Steps: {df["Steps"].mean():,.0f}')
    print(f'   Best Day: {df["Steps"].max():,} steps')
    print(f'   Active Days (30+ min exercise): {df["Active_Day"].sum()} ({df["Active_Day"].mean()*100:.1f}%)')
    
    # Health metrics
    print(f'\n💪 Health Metrics:')
    print(f'   Average Sleep: {df["Sleep_Hours"].mean():.1f} hours')
    print(f'   Days with Good Sleep: {df["Good_Sleep"].sum()} ({df["Good_Sleep"].mean()*100:.1f}%)')
    print(f'   Well Hydrated Days: {df["Well_Hydrated"].sum()} ({df["Well_Hydrated"].mean()*100:.1f}%)')
    
    # Weight progress
    print(f'\n⚖️  Weight Progress:')
    print(f'   Starting: {df["Weight_lbs"].iloc[0]:.1f} lbs')
    print(f'   Current: {df["Weight_lbs"].iloc[-1]:.1f} lbs')
    print(f'   Change: {df["Weight_lbs"].iloc[-1] - df["Weight_lbs"].iloc[0]:.1f} lbs')
    
    # Day of week patterns
    print(f'\n📅 Best Day of Week for Activity:')
    dow_steps = df.groupby('DayOfWeek')['Steps'].mean().sort_values(ascending=False)
    print(f'   {dow_steps.index[0]}: {dow_steps.iloc[0]:,.0f} steps')
    
    print('='*60)

generate_fitness_report(df)

## Step 5: Visualizations

In [ ]:
# Create comprehensive dashboard
fig = plt.figure(figsize=(16, 12))

# 1. Steps Over Time
ax1 = plt.subplot(3, 2, 1)
ax1.plot(df['Date'], df['Steps'], marker='o', linewidth=2, color='#4ECDC4')
ax1.axhline(10000, color='red', linestyle='--', label='10K Goal')
ax1.set_title('Daily Steps Trend', fontsize=14, fontweight='bold')
ax1.set_xlabel('Date')
ax1.set_ylabel('Steps')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# 2. Weight Progress
ax2 = plt.subplot(3, 2, 2)
ax2.plot(df['Date'], df['Weight_lbs'], marker='o', linewidth=2, color='#FF6B6B')
ax2.set_title('Weight Trend', fontsize=14, fontweight='bold')
ax2.set_xlabel('Date')
ax2.set_ylabel('Weight (lbs)')
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

# 3. Exercise vs Sleep
ax3 = plt.subplot(3, 2, 3)
ax3.scatter(df['Exercise_Minutes'], df['Sleep_Hours'], s=100, alpha=0.6, c='green')
ax3.set_title('Exercise vs Sleep Quality', fontsize=14, fontweight='bold')
ax3.set_xlabel('Exercise (minutes)')
ax3.set_ylabel('Sleep (hours)')
ax3.grid(True, alpha=0.3)

# 4. Steps Categories
ax4 = plt.subplot(3, 2, 4)
steps_counts = df['Steps_Category'].value_counts()
ax4.bar(steps_counts.index, steps_counts.values, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
ax4.set_title('Steps Performance Distribution', fontsize=14, fontweight='bold')
ax4.set_ylabel('Number of Days')

# 5. Day of Week Analysis
ax5 = plt.subplot(3, 2, 5)
dow_avg = df.groupby('DayOfWeek')['Steps'].mean().reindex(
    ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
)
ax5.bar(range(len(dow_avg)), dow_avg.values, color='purple', alpha=0.7)
ax5.set_xticks(range(len(dow_avg)))
ax5.set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
ax5.set_title('Average Steps by Day of Week', fontsize=14, fontweight='bold')
ax5.set_ylabel('Average Steps')

# 6. Metrics Summary
ax6 = plt.subplot(3, 2, 6)
metrics = {
    'Active Days': df['Active_Day'].sum(),
    'Good Sleep Days': df['Good_Sleep'].sum(),
    'Well Hydrated': df['Well_Hydrated'].sum()
}
ax6.bar(metrics.keys(), metrics.values(), color=['#90EE90', '#87CEEB', '#DDA0DD'])
ax6.set_title('Health Goals Achievement', fontsize=14, fontweight='bold')
ax6.set_ylabel('Number of Days')
ax6.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('fitness_complete_dashboard.png', dpi=300, bbox_inches='tight')
plt.show()

print('Complete dashboard saved!')

## Step 6: Export Results

In [ ]:
# Save cleaned data
df.to_csv('fitness_analysis.csv', index=False)
print('Data saved to fitness_analysis.csv')

# Create summary report
summary = {
    'Metric': ['Average Steps', 'Active Days', 'Average Sleep', 'Weight Change'],
    'Value': [
        f"{df['Steps'].mean():,.0f}",
        f"{df['Active_Day'].sum()} / {len(df)}",
        f"{df['Sleep_Hours'].mean():.1f} hours",
        f"{df['Weight_lbs'].iloc[-1] - df['Weight_lbs'].iloc[0]:.1f} lbs"
    ]
}
summary_df = pd.DataFrame(summary)
summary_df.to_csv('fitness_summary.csv', index=False)
print('Summary saved to fitness_summary.csv')

## Your Own Project

Now it's your turn! Choose a dataset that interests you and apply all the skills you've learned.

In [ ]:
# Your project code here


## What's Next?

### Continue Learning:
- **APIs**: Learn to fetch data from web services
- **Web Scraping**: Extract data from websites (BeautifulSoup)
- **Advanced Pandas**: Pivot tables, time series
- **Statistical Analysis**: scipy, statsmodels
- **Machine Learning**: scikit-learn basics
- **Web Apps**: Build dashboards with Streamlit
- **Automation**: Schedule Python scripts

### Practice Resources:
- Kaggle datasets and competitions
- Real Python tutorials
- DataCamp courses
- Python documentation
- GitHub projects

### Build Projects:
- Personal budget tracker
- Weather data analyzer
- Social media sentiment analysis
- Automated report generator
- Data visualization dashboard

Congratulations on completing the course! 🎉

## Course Summary

**Session 1**: Variables & Calculator
**Session 2**: Strings & Text
**Session 3**: Lists & Collections
**Session 4**: Loops & Iteration
**Session 5**: Conditionals
**Session 6**: Dictionaries
**Session 7**: Functions
**Session 8**: File Handling
**Session 9**: Pandas Basics
**Session 10**: Data Cleaning
**Session 11**: Visualization
**Session 12**: Complete Project

You now have practical Python skills for data analysis, automation, and real-world problem solving!